# 06. Training orchestrator

`train()` is the outer AlphaZero loop. Each iteration it runs one self-play
episode (component 03 search, guided by the component 05 neural evaluator),
pushes the trajectory into a replay buffer, and takes a few gradient steps on the
component 04 network. It returns a `TrainingLog` with per-iteration losses and the
best formula seen.

`play()` is the trained policy's own answer: a greedy MCTS rollout using the
trained network, with no reward-max post-walk. `play_top_k` runs it several times,
forbidding each previous first move, for a diverse shortlist.

The simulator (the reward source) is injected, so `train()` has no environment
baked in. It needs the `[nn]` extra (`torch`).

In [1]:
import alpha_rule
import torch
print("alpha_rule", alpha_rule.__version__, "| torch", torch.__version__)

alpha_rule 0.1.0 | torch 2.11.0+cu128


## Elements

`alpha_rule.training`
* `train(grammar, expensive_simulator, ...)`: the self-play + replay + NN-update
  loop. Returns a `TrainingLog`. Every knob (search, buffer, network, optimiser,
  selection, backup, exploration) is a keyword argument with a default.
* `play(log, grammar, simulator, ...)` and `play_top_k(...)`: greedy rollouts with
  the trained network.
* `TrainingLog` / `IterationLog`: the per-run and per-iteration records.
* `AlphaZeroCSVLogger` / `RunLevelEvalLogger`: optional CSV logging via the
  `on_iteration_end` hook; `load_alphazero_logs` reads them back (pandas).

`value_scale` defaults to the simulator's `reward_scale` (via
`NeuralEvaluator.from_simulator`), so the network value and the simulator reward
share one scale in the MCTS backup.

## A training run

A tiny, fast run on a toy reward: rules score by the fraction of their tokens that
are `A` or `<` (in `[0, 1]`, so `reward_scale = 1.0`). The loop should run, the
losses should stay finite, and the best formula should lean toward `A` and `<`.

In [2]:
import numpy as np
from alpha_rule.grammar.allen import AllenIntervalGrammar
from alpha_rule.training import train

class PreferenceSimulator:
    # Reward is the fraction of preferred tokens (A, <), in [0, 1].
    reward_scale = 1.0
    def evaluate(self, node):
        toks = node.name.replace("<END>", "").split()
        if not toks:
            return 0.0
        return sum(t in ("A", "<") for t in toks) / len(toks)

g = AllenIntervalGrammar(event_types=("A", "B"), relations=("<", ">"))
log = train(
    grammar=g,
    expensive_simulator=PreferenceSimulator(),
    n_iterations=12,
    n_simulations=12,
    depth_limit=4,
    buffer_warmup=4,
    batch_size=8,
    train_steps_per_iteration=2,
    d_model=24, nhead=2, num_layers=1, max_len=32,
    learning_rate=5e-3, weight_decay=1e-4,
    seed=0, device="cpu",
)
print("iterations       :", len(log.iterations))
print("device           :", log.device)
print("value_scale      :", log.value_scale, "(read from simulator.reward_scale)")
print("best formula      :", repr(log.best_formula), "reward", round(log.best_reward, 3))
last = log.iterations[-1]
print("final losses      : total", round(last.train_total, 3),
      "policy", round(last.train_policy, 3), "value", round(last.train_value, 3))

iterations       : 12
device           : cpu
value_scale      : 1.0 (read from simulator.reward_scale)
best formula      : 'A A < <END>' reward 1.0
final losses      : total 0.767 policy 0.766 value 0.001


## The policy's answer (play)

`play()` runs one temperature-0 (argmax) MCTS rollout with the trained network and
returns the constructed rule and its reward. `play_top_k` repeats it, forbidding
each previous first move, for a diverse shortlist.

In [3]:
from alpha_rule.training import play, play_top_k

rule, reward = play(log, grammar=g, simulator=PreferenceSimulator())
print("play() rule  :", repr(rule), "reward", round(reward, 3))

print("play_top_k(k=3):")
for r, rw in play_top_k(log, grammar=g, simulator=PreferenceSimulator(), k=3):
    print(f"  {r!r:16} reward {rw:.3f}")

play() rule  : 'A A < A' reward 1.0
play_top_k(k=3):
  'A A < A'        reward 1.000
  'B A < A'        reward 0.750


## Logging

`AlphaZeroCSVLogger` writes one CSV row per iteration. Wire it through
`train(on_iteration_end=logger.as_on_iteration_end())` for a live run, or replay a
finished `TrainingLog` in one shot as below.

In [4]:
import os
import tempfile
from alpha_rule.training import AlphaZeroCSVLogger

with tempfile.TemporaryDirectory() as d:
    logger = AlphaZeroCSVLogger(base_dir=d, env_name="demo", activity="notebook",
                                strategy="PUCT+Max")
    logger.log_training_log(log)
    lines = open(logger.csv_path, encoding="utf-8").read().splitlines()
    print("file       :", os.path.basename(logger.csv_path))
    print("data rows  :", len(lines) - 1)
    print("columns    :", lines[0].split(",")[:6], "...")

file       : demo_az_PUCT+Max_0_results.csv
data rows  : 12
columns    : ['iteration', 'wall_time_s', 'policy_loss', 'value_loss', 'total_loss', 'best_reward_in_trajectory'] ...


## Basic checks

Asserts mirroring the unit tests. `ok` means the loop wires up.

In [5]:
import math
import numpy as np
from alpha_rule.grammar.allen import AllenIntervalGrammar
from alpha_rule.training import train, play, TrainingLog

class _Sim:
    reward_scale = 2.0
    def evaluate(self, node):
        return 1.0

g = AllenIntervalGrammar(event_types=("A", "B"), relations=("<",))
log = train(grammar=g, expensive_simulator=_Sim(), n_iterations=3, n_simulations=8,
            depth_limit=3, buffer_warmup=1, batch_size=4, train_steps_per_iteration=1,
            d_model=16, nhead=2, num_layers=1, max_len=16, seed=0, device="cpu")

assert isinstance(log, TrainingLog) and len(log.iterations) == 3
assert log.model is not None and log.max_len == 16
assert log.value_scale == 2.0                       # wired from simulator.reward_scale
assert all(math.isfinite(it.train_total) for it in log.iterations)

rule, reward = play(log, grammar=g, simulator=_Sim())
assert isinstance(rule, str) and len(rule) > 0

print("ok")

ok


## Full unit tests

```
python -m alpha_rule.tests.run_tests test_train_orchestrator
python -m alpha_rule.tests.run_tests test_play
python -m alpha_rule.tests.run_tests test_train_hyperparameters
python -m alpha_rule.tests.run_tests test_alphazero_csv_logger
python -m alpha_rule.tests.run_tests test_run_level_eval_logger
```

These need the `[nn]` extra (`torch`). The full `train()` + RL-eval integration
test arrives with the reinforcement-learning backend (component 07).